# 01. Scala benchmark: từ GraphX API tới bằng chứng hiệu năng

Notebook này chạy trực tiếp bằng kernel **Scala 2.12.18 - GraphX benchmark**.
Không nhúng Scala vào chuỗi Python và không sinh JAR để chạy thuật toán.
Notebook `02_python_visualize.ipynb` sẽ đọc output của lần chạy thành công để giải thích kết quả.

Luồng đọc: cấu hình → input chung → GraphX → GraphFrames → kiểm tra đúng → đo lặp → xuất kết quả.
Phần GraphX theo [GraphX Programming Guide 4.0.2](https://spark.apache.org/docs/4.0.2/graphx-programming-guide.html).
Môi trường thực thi được giữ ở **Spark 3.5.7, Scala 2.12.18, Java 17, GraphFrames 0.10.0**; các public API sử dụng có cùng cách gọi trong guide.

Chạy `scripts/setup_scala_kernel.py` bằng Python trong `.venv` để chuẩn bị kernel, sau đó chọn kernel Scala và Run All.
Khi chạy lại, restart kernel trước để có một SparkContext mới và một event log mới.

## 1. Cấu hình một thí nghiệm có thể đối chiếu

Mặc định dùng 50.000 cạnh nhận bằng USD, 4 partition, 5 vòng PageRank và 3 repeat sau một warmup cho mỗi cấu hình.
Đặt `sampleEdges = 0` để sử dụng toàn bộ nhóm nhận USD và tăng heap của kernel nếu cần.
Mẫu cạnh không giữ nguyên topology của graph đầy đủ, nên kết luận chỉ áp dụng cho input thực nghiệm.
Không nâng Spark chỉ để đổi cú pháp API: toàn bộ môi trường được ghi lại cùng output.

In [1]:
import java.io.{File, PrintWriter}
import java.nio.file.Files
import java.lang.management.ManagementFactory
import java.security.MessageDigest
import scala.collection.JavaConverters._

val root = Iterator.iterate(new File(".").getCanonicalFile)(_.getParentFile)
  .takeWhile(_ != null)
  .find(dir => new File(dir, "scripts/download_aml.py").isFile).get
val sampleEdges = 50000
val p = 4
val iterations = 5
val repeats = 3
val warmups = 1
val seed = 42L
val damping = 0.85
val master = "local[2]"
val currency = "US Dollar"
val reportRoot = new File(root, "reports/architecture")
val output = new File(reportRoot, "runs/scala_" + System.currentTimeMillis())
val eventDir = new File(output, "events")
val inputDir = new File(root, "data/benchmark/" + output.getName)
eventDir.mkdirs()
inputDir.mkdirs()
require(p >= 2 && iterations > 0 && repeats >= 2 && warmups >= 1)
println(s"Run: ${output.getAbsolutePath}")

Run: /home/tuanta/graphX/reports/architecture/runs/scala_1792035232036


### Nạp Spark và GraphFrames

Các JAR được nạp từ môi trường đã cài trong dự án để kernel và Spark dùng cùng dependency.
`almond-spark` hỗ trợ chạy Spark tương tác và gửi class tạo trong notebook tới executor.
Phần này chỉ là thiết lập môi trường, không nằm trong benchmark.

In [2]:
interp.load.cp {
  val sparkHome = os.Path(sys.env("SPARK_HOME"))
  os.list(sparkHome / "jars").filter(_.ext == "jar").filterNot { jar =>
    Seq("scala-library-", "scala-reflect-", "scala-compiler-", "spark-repl_")
      .exists(jar.last.startsWith)
  }
}

In [3]:
import $ivy.`sh.almond::almond-spark:0.14.0-RC8`
interp.load.cp(Seq(
  os.Path(new File(root, ".tools/jars/graphframes-spark3_2.12-0.10.0.jar")),
  os.Path(new File(root, ".tools/jars/graphframes-graphx-spark3_2.12-0.10.0.jar"))
))

In [4]:
import org.apache.spark.sql._
import org.apache.spark.sql.functions._
import org.apache.spark.graphx._
import org.apache.spark.rdd.RDD
import org.apache.spark.storage.StorageLevel
import org.graphframes.GraphFrame
import org.graphframes.lib.AggregateMessages

val spark = NotebookSparkSession.builder()
  .master(master).appName("GraphX-vs-GraphFrames-native-Scala")
  .config("spark.eventLog.enabled", "true")
  .config("spark.eventLog.compress", "false")
  .config("spark.eventLog.dir", eventDir.toURI.toString)
  .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
  .config("spark.sql.shuffle.partitions", p)
  .config("spark.sql.autoBroadcastJoinThreshold", -1)
  .config("spark.sql.adaptive.autoBroadcastJoinThreshold", -1)
  .config("spark.sql.adaptive.advisoryPartitionSizeInBytes", 1048576)
  .config("spark.sql.adaptive.coalescePartitions.parallelismFirst", false)
  .getOrCreate()
val sc = spark.sparkContext
import spark.implicits._
val level = StorageLevel.MEMORY_AND_DISK
sc.setCheckpointDir(new File(output, "checkpoints").getAbsolutePath)
sc.setLogLevel("ERROR")
require(spark.version == "3.5.7")
require(scala.util.Properties.versionNumberString == "2.12.18")
println(s"Spark ${spark.version}; Scala ${scala.util.Properties.versionNumberString}; UI ${sc.uiWebUrl}")

Loading spark-stubs

Failed to download https://repo.maven.apache.org/maven2/org/apache/hadoop/hadoop-client-runtime/3.3.1/hadoop-client-runtime-3.3.1-sources.jar


Failed to download https://repo.maven.apache.org/maven2/org/apache/hadoop/hadoop-client-runtime/3.3.1/hadoop-client-runtime-3.3.1-sources.jar.sha1


Creating SparkSession

Spark UI

Spark 3.5.7; Scala 2.12.18; UI Some(http://localhost:4040)


## 2. Chuẩn hóa một input chung

Số tiền là `Amount Received` và chỉ lấy các giao dịch có `Receiving Currency = US Dollar`.
Giữ self-loop và cạnh song song; mỗi giao dịch là một cạnh có hướng.
Spark chọn mẫu ngẫu nhiên trên toàn bộ nhóm USD, không lấy các dòng đầu CSV.
Seed gắn với input partitioning; manifest và hai tập Parquet xác định chính xác input của lần chạy.
Chi phí đọc CSV, lấy mẫu và chuẩn hóa nằm ngoài mọi timer benchmark.

In [5]:
val rawPath = new File(root, "data/raw/HI-Small_Trans.csv")
require(rawPath.isFile, "Chạy scripts/download_aml.py trước.")
val raw = spark.read.option("header", true).option("inferSchema", false)
  .csv(rawPath.getAbsolutePath)
  .toDF("timestamp", "from_bank", "from_account", "to_bank", "to_account",
    "amount_received", "receiving_currency", "amount_paid", "payment_currency",
    "payment_format", "is_laundering")
val eligible = raw.filter($"receiving_currency" === currency)
val eligibleCount = eligible.count()
val sampled = if (sampleEdges == 0) eligible else
  eligible.orderBy(rand(seed)).limit(sampleEdges)
val selected = sampled.select(
  monotonically_increasing_id().as("edge_id"),
  $"from_bank", $"from_account", $"to_bank", $"to_account",
  $"amount_received".cast("double").as("amount"), $"timestamp"
).persist(level)
val selectedCount = selected.count()
require(selectedCount > 0)
require(selected.filter($"amount".isNull || isnan($"amount") || abs($"amount") === Double.PositiveInfinity).count() == 0)
require(selected.filter($"from_bank".isNull || $"from_account".isNull || $"to_bank".isNull || $"to_account".isNull).count() == 0)
println(s"Eligible USD: $eligibleCount; selected: $selectedCount")

csv at cmd5.sc:4

count at cmd5.sc:9

count at cmd5.sc:9

count at cmd5.sc:17

count at cmd5.sc:17

count at cmd5.sc:17

count at cmd5.sc:19

count at cmd5.sc:20

Eligible USD: 1879341; selected: 50000


### Vertex ID ổn định và không va chạm

Mỗi node là cặp `(bank, account)`, giữ nguyên các số 0 đầu trong chuỗi gốc.
Sắp xếp các cặp khác nhau rồi đánh số Long từ 1 bằng `zipWithIndex`, không dùng hash làm ID.
Cả GraphX và GraphFrames sẽ đọc lại cùng `vertices` và `edges` từ hai tập Parquet bên dưới.

In [6]:
val accounts = selected.select($"from_bank".as("bank"), $"from_account".as("account"))
  .union(selected.select($"to_bank".as("bank"), $"to_account".as("account")))
  .distinct().orderBy("bank", "account")
val vertexData = accounts.as[(String, String)].rdd.zipWithIndex()
  .map { case ((bank, account), index) => (index + 1L, bank, account) }
  .toDF("id", "bank", "account")
val senderIds = vertexData.select($"id".as("src"), $"bank".as("from_bank"), $"account".as("from_account"))
val receiverIds = vertexData.select($"id".as("dst"), $"bank".as("to_bank"), $"account".as("to_account"))
val edgeData = selected.join(senderIds, Seq("from_bank", "from_account"))
  .join(receiverIds, Seq("to_bank", "to_account"))
  .select("edge_id", "src", "dst", "amount", "timestamp")
vertexData.repartition(p).write.parquet(new File(inputDir, "vertices").getAbsolutePath)
edgeData.repartition(p).write.parquet(new File(inputDir, "edges").getAbsolutePath)
selected.unpersist(blocking = true)

val vertices = spark.read.parquet(new File(inputDir, "vertices").getAbsolutePath).persist(level)
val edges = spark.read.parquet(new File(inputDir, "edges").getAbsolutePath).persist(level)
val n = vertices.count()
val edgeCount = edges.count()
require(n > 0 && edgeCount == selectedCount)
require(vertices.select("id").distinct().count() == n)
require(edges.select("edge_id").distinct().count() == edgeCount)
val baseCacheIds = sc.getPersistentRDDs.keySet.toSet
vertices.show(5, false)
edges.show(5, false)
println(s"Common input: $n vertices, $edgeCount edges")

rdd at cmd6.sc:4

rdd at cmd6.sc:4

rdd at cmd6.sc:4

parquet at cmd6.sc:12

parquet at cmd6.sc:12

...

show at cmd6.sc:24

+-----+-------+---------+
|id   |bank   |account  |
+-----+-------+---------+
|26081|0121358|81329CCF0|
|39802|021174 |8082E05B0|
|29883|0132923|80D41AE70|
|10853|003051 |802761CC0|
|7000 |0019890|807BA7600|
+-----+-------+---------+
only showing top 5 rows



show at cmd6.sc:25

+-------+-----+-----+---------+----------------+
|edge_id|src  |dst  |amount   |timestamp       |
+-------+-----+-----+---------+----------------+
|48675  |2649 |2649 |160524.66|2022/09/01 20:06|
|15392  |46430|50193|1692.04  |2022/09/02 01:56|
|46061  |16564|28424|3194.32  |2022/09/09 06:14|
|10126  |17621|8864 |3791.72  |2022/09/07 01:58|
|46931  |15268|15268|3.02     |2022/09/01 02:25|
+-------+-----+-----+---------+----------------+
only showing top 5 rows



Common input: 54019 vertices, 50000 edges


## 3. Ranh giới phép đo

`build` gồm tạo representation, repartition và materialize cache.
`aggregate` gồm gửi message, cộng và materialize kết quả.
`pr_setup` gồm degree, trọng số cạnh và checkpoint chuẩn bị; mỗi `pr_iteration` gồm dangling reduction, message, update và checkpoint state.
Các chức năng đo thời gian dưới đây chỉ bọc bên ngoài thuật toán, không thay thế GraphX API.
Cache được đo bằng Spark cached blocks, không phải tổng RSS hoặc peak heap.

In [7]:
val metrics = new PrintWriter(new File(output, "phase_timings.csv"))
val checks = new PrintWriter(new File(output, "correctness.csv"))
val loads = new PrintWriter(new File(output, "partition_load.csv"))
val progress = new PrintWriter(new File(output, "progress.log"))
var currentRun = ""
var expectedIncoming = Map.empty[Long, Double]
var expectedRanks = Map.empty[Long, Double]

expectedRanks: Map[Long, Double] = Map()

In [8]:
def csv(writer: PrintWriter, fields: Seq[Any]): Unit = {
    writer.println(fields.map(x => "\"" + x.toString.replace("\"", "\"\"") + "\"").mkString(","))
    writer.flush()
  }
  def gcMillis: Long = ManagementFactory.getGarbageCollectorMXBeans.asScala.map(_.getCollectionTime).filter(_ >= 0).sum
  def cacheBytes: Long = sc.getRDDStorageInfo.map(_.memSize).sum
  def diskBytes: Long = sc.getRDDStorageInfo.map(_.diskSize).sum
  def measure[A](phase: String, iteration: Int = 0)(body: => A): A = {
    val group = s"$currentRun|$phase|$iteration"
    sc.setJobGroup(group, group)
    val started = System.nanoTime()
    val gcStart = gcMillis
    val memStart = cacheBytes
    val diskStart = diskBytes
    val result = body
    val elapsed = (System.nanoTime() - started) / 1e6
    csv(metrics, Seq(currentRun, phase, iteration, elapsed, cacheBytes - memStart, cacheBytes, diskBytes - diskStart, gcMillis - gcStart))
    sc.clearJobGroup()
    progress.println(f"BENCH $group wall_ms=$elapsed%.1f"); progress.flush()
    result
  }
  def writeText(name: String, text: String): Unit = {
    val w = new PrintWriter(new File(output, name)); try w.println(text) finally w.close()
  }
  def validateResult(kind: String, result: Map[Long, Double], expected: Map[Long, Double]): Unit = {
    val error = expected.keysIterator.map { id =>
      math.abs(result.getOrElse(id, 0.0) - expected(id)) / math.max(1.0, math.abs(expected(id)))
    }.foldLeft(0.0)(math.max)
    require(result.keySet.subsetOf(expected.keySet), s"Unexpected IDs in $kind")
    if (kind == "pagerank") require(result.size == expected.size, "Missing PageRank vertices")
    val absolute = expected.keysIterator.map(id => math.abs(result.getOrElse(id, 0.0) - expected(id))).foldLeft(0.0)(math.max)
    val mass = result.values.sum
    require(error < 1e-9, s"$currentRun $kind relative-scaled error=$error")
    if (kind == "pagerank") require(math.abs(mass - 1.0) < 1e-9, s"Rank mass=$mass")
    csv(checks, Seq(currentRun, kind, result.size, absolute, error, mass, true))
  }
  def cleanup(): Unit = {
    sc.getPersistentRDDs.filterNot { case (id, _) => baseCacheIds(id) }.values.foreach(_.unpersist(blocking = true))
  }
  def saveLoad(counts: Seq[(Int, Long)]): Unit = {
    val actual = counts.toMap
    (0 until p).foreach(i => csv(loads, Seq(currentRun, i, actual.getOrElse(i, 0L))))
  }

In [9]:
csv(metrics, Seq("run", "phase", "iteration", "wall_ms", "cache_delta_bytes", "cache_total_bytes", "disk_delta_bytes", "jvm_gc_ms"))
csv(checks, Seq("run", "kind", "vertices_returned", "max_abs_error", "max_scaled_error", "sum", "passed"))
csv(loads, Seq("run", "partition_id", "edges"))

## 4. GraphX: bắt đầu từ Graph, VertexRDD và EdgeRDD

`Graph(vertexRDD, edgeRDD, defaultValue)` là constructor chính trong guide.
Các tham số storage level thêm vào nhằm giữ cùng cache policy với GraphFrames.
Không tạo một RDD join pipeline để giả làm GraphX; mọi phép tính bên dưới dùng graph operators chính thức.

In [10]:
// Same public GraphX API as the programming guide.
  def buildGraphX(
      vertexData: DataFrame,
      edgeData: DataFrame,
      strategy: PartitionStrategy,
      partitions: Int): Graph[Double, Double] = {

    val vertexRDD: RDD[(VertexId, Double)] = vertexData.select("id").as[Long].rdd
      .map(id => (id, 1.0))

    val edgeRDD: RDD[Edge[Double]] = edgeData.select("src", "dst", "amount")
      .as[(Long, Long, Double)].rdd
      .map { case (src, dst, amount) => Edge(src, dst, amount) }

    val graph = Graph(
      vertexRDD,
      edgeRDD,
      defaultVertexAttr = 1.0,
      edgeStorageLevel = level,
      vertexStorageLevel = level)

    graph.partitionBy(strategy, partitions).persist(level)
  }

### Incoming amount: gửi một message cho mỗi cạnh

Message là số tiền trên cạnh, gửi tới `dst` bằng `sendToDst`.
Hàm merge chỉ cộng hai số Double.
`TripletFields.None` cho biết không cần lấy thuộc tính vertex.

In [11]:
def incomingAmounts(graph: Graph[Double, Double]): VertexRDD[Double] = {
    graph.aggregateMessages[Double](
      sendMsg = edge => edge.sendToDst(edge.attr),
      mergeMsg = (left, right) => left + right,
      tripletFields = TripletFields.None)
  }

### PageRank: thiết lập state bằng property operators

Với N vertex, d = 0.85, out-degree k(u) đếm cả cạnh song song và dangling mass D(t):

\[
r_v^{(t+1)} = (1-d)/N + d\left(\sum_{u\to v}r_u^{(t)}/k(u) + D^{(t)}/N\right).
\]

Rank khởi tạo bằng 1/N; tổng rank phải xấp xỉ 1 sau mỗi vòng.
Theo pattern trong guide, `outerJoinVertices` bổ sung degree, `mapTriplets` tạo trọng số cạnh và `mapVertices` khởi tạo rank.
State là `Graph[Double, Double]`, không cần tuple rank/degree trong mỗi vòng.
Danh sách dangling vertex được giữ riêng để tránh tính lại degree.

In [12]:
def preparePageRank(
      graph: Graph[Double, Double],
      vertexCount: Long): (Graph[Double, Double], VertexRDD[Int]) = {

    val degreeGraph = graph.outerJoinVertices(graph.outDegrees) {
      (_, _, degree) => degree.getOrElse(0)
    }.persist(level)
    degreeGraph.vertices.count()

    val danglingVertices = degreeGraph.vertices.filter {
      case (_, outDegree) => outDegree == 0
    }.persist(level)
    danglingVertices.count()

    // Exactly the guide's mapTriplets -> mapVertices pattern.
    // Each edge carries 1 / source out-degree; each vertex carries its rank.
    val rankGraph = degreeGraph
      .mapTriplets(edge => 1.0 / edge.srcAttr, TripletFields.Src)
      .mapVertices((_, _) => 1.0 / vertexCount)
      .persist(level)

    rankGraph.vertices.checkpoint()
    rankGraph.edges.checkpoint()
    rankGraph.vertices.count()
    rankGraph.edges.count()

    (rankGraph, danglingVertices)
  }

### Một vòng lặp chỉ có ba công việc

Tính dangling mass, gửi rank contribution dọc cạnh và cập nhật tất cả vertex bằng `outerJoinVertices`.
Vertex không nhận message vẫn có teleport và dangling contribution.
Không dùng PageRank built-in vì benchmark cần cùng công thức và đúng số vòng với GraphFrames.

In [13]:
def pageRankStep(
      rankGraph: Graph[Double, Double],
      danglingVertices: VertexRDD[Int],
      vertexCount: Long,
      damping: Double): Graph[Double, Double] = {

    val danglingMass = rankGraph.vertices.innerJoin(danglingVertices) {
      (_, rank, _) => rank
    }.values.sum()

    val messages = rankGraph.aggregateMessages[Double](
      sendMsg = edge => edge.sendToDst(edge.srcAttr * edge.attr),
      mergeMsg = (left, right) => left + right,
      tripletFields = TripletFields.Src)

    rankGraph.outerJoinVertices(messages) { (_, _, incoming) =>
      (1.0 - damping) / vertexCount +
        damping * (incoming.getOrElse(0.0) + danglingMass / vertexCount)
    }
  }

### Chạy GraphX với các timer đã định nghĩa

Algorithm ở trên không chứa timer hoặc mã xuất CSV.
Đo đạc dưới đây chỉ gọi thuật toán và force action để tránh đo lazy plan thay vì tính toán thật.
Checkpoint mỗi vòng nằm trong timing và có thể ảnh hưởng chi phí tái sử dụng vertex index; đây là một giới hạn cần đọc cùng kết quả.

In [14]:
// Timing, validation and evidence collection are separate from the algorithms.
  def runGraphX(strategyName: String, saveEvidence: Boolean): Unit = {
    val strategy = strategyName match {
      case "RandomVertexCut" => PartitionStrategy.RandomVertexCut
      case "EdgePartition1D" => PartitionStrategy.EdgePartition1D
      case "EdgePartition2D" => PartitionStrategy.EdgePartition2D
      case "CanonicalRandomVertexCut" => PartitionStrategy.CanonicalRandomVertexCut
    }
    val graph = measure("build") {
      val result = buildGraphX(vertices, edges, strategy, p)
      require(result.vertices.count() == n)
      require(result.edges.count() == edgeCount)
      result
    }
    if (saveEvidence) {
      writeText(currentRun + "_rdd_dag.txt", graph.vertices.toDebugString + "\n\n" + graph.edges.toDebugString)
      writeText(currentRun + "_representation.txt", s"vertices=${graph.vertices.getClass.getName}\nedges=${graph.edges.getClass.getName}\nlevel=${graph.vertices.getStorageLevel}")
    }
    saveLoad(graph.edges.mapPartitionsWithIndex { (i, it) => Iterator((i, it.size.toLong)) }.collect().toSeq)

    val incoming = measure("aggregate") {
      val result = incomingAmounts(graph).persist(level)
      result.count()
      result.values.sum()
      result
    }
    validateResult("incoming", incoming.collect().toMap, expectedIncoming)
    incoming.unpersist(blocking = true)

    val (initialRankGraph, danglingVertices) = measure("pr_setup") {
      preparePageRank(graph, n)
    }
    var rankGraph = initialRankGraph
    (1 to iterations).foreach { iteration =>
      rankGraph = measure("pr_iteration", iteration) {
        val nextGraph = pageRankStep(rankGraph, danglingVertices, n, damping)
        nextGraph.vertices.persist(level)
        nextGraph.vertices.checkpoint()
        val mass = nextGraph.vertices.values.sum()
        require(math.abs(mass - 1.0) < 1e-9, s"PageRank mass=$mass")
        if (saveEvidence && iteration == 1) {
          writeText(currentRun + "_pr_rdd_dag.txt", nextGraph.vertices.toDebugString)
        }
        rankGraph.unpersistVertices(blocking = true)
        nextGraph
      }
    }
    validateResult("pagerank", rankGraph.vertices.collect().toMap, expectedRanks)
    danglingVertices.unpersist(blocking = true)
  }

## 5. GraphFrames: cùng logic trên DataFrame

Vertex và cạnh vẫn lấy từ input chung, nhưng graph được xây bằng DataFrame có schema.
Số tiền nhận và công thức PageRank hoàn toàn giống GraphX.
Chỉ thay key repartition src/dst và AQE ON/OFF.
Ở GraphFrames 0.10.0, `aggregateMessages.agg()` tự materialize và persist output, nên lời gọi này phải nằm trong timer.
Checkpoint state và weighted edges tách lineage, tránh self-join resolution trên cùng source plan.

In [15]:
def runGraphFrames(key: String, aqe: Boolean, saveEvidence: Boolean): Unit = {
    spark.conf.set("spark.sql.adaptive.enabled", aqe.toString)
    val g = measure("build") {
      val v = vertices.select($"id", lit(1.0).as("value")).repartition(p, $"id").persist(level)
      val e = edges.select("src", "dst", "amount").repartition(p, col(key)).persist(level)
      val graph = GraphFrame(v, e)
      require(graph.vertices.count() == n)
      require(graph.edges.count() == edgeCount)
      graph
    }
    if (saveEvidence) {
      writeText(currentRun + "_representation.txt", s"vertices=${g.vertices.getClass.getName}\nedges=${g.edges.getClass.getName}\nlevel=${g.vertices.storageLevel}")
    }
    saveLoad(g.edges.groupBy(spark_partition_id().as("partition_id")).count().as[(Int, Long)].collect().toSeq)
    val incoming = measure("aggregate") {
      val query = g.aggregateMessages.sendToDst(AggregateMessages.edge("amount")).agg(sum(AggregateMessages.msg).as("incoming"))
      if (saveEvidence) writeText(currentRun + "_aggregate_plan_before.txt", query.queryExecution.toString)
      query.persist(level)
      query.count()
      query.agg(sum("incoming")).first()
      query
    }
    if (saveEvidence) writeText(currentRun + "_aggregate_plan_after.txt", incoming.queryExecution.executedPlan.toString)
    validateResult("incoming", incoming.select("id", "incoming").as[(Long, Double)].collect().toMap, expectedIncoming)
    incoming.unpersist(blocking = true)

    var state: DataFrame = null
    val weightedEdges = measure("pr_setup") {
      val degree = g.edges.groupBy("src").count().select($"src".as("id"), $"count".as("out_degree"))
      state = g.vertices.select("id").join(degree, Seq("id"), "left").na.fill(0L, Seq("out_degree"))
        .withColumn("rank", lit(1.0 / n)).checkpoint(eager = true).persist(level)
      state.count()
      val weighted = g.edges.join(state.select($"id".as("src"), $"out_degree"), Seq("src"))
        .select($"src", $"dst", (lit(1.0) / $"out_degree").as("weight")).repartition(p, col(key)).checkpoint(eager = true).persist(level)
      weighted.count()
      weighted
    }
    (1 to iterations).foreach { iteration =>
      state = measure("pr_iteration", iteration) {
        val rawDangling = state.filter($"out_degree" === 0L).agg(sum("rank")).first()
        val dangling = if (rawDangling.isNullAt(0)) 0.0 else rawDangling.getDouble(0)
        val currentGraph = GraphFrame(state, weightedEdges)
        val messages = currentGraph.aggregateMessages
          .sendToDst(AggregateMessages.src("rank") * AggregateMessages.edge("weight"))
          .agg(sum(AggregateMessages.msg).as("message"))
        val update = state.select("id", "out_degree").join(messages, Seq("id"), "left")
          .select($"id", $"out_degree", (lit((1.0 - damping) / n) + lit(damping) * (coalesce($"message", lit(0.0)) + lit(dangling / n))).as("rank"))
        val next = update.checkpoint(eager = false).persist(level)
        val mass = next.agg(sum("rank")).first().getDouble(0)
        require(math.abs(mass - 1.0) < 1e-9, s"PageRank mass=$mass")
        if (saveEvidence && iteration == 1) {
          writeText(currentRun + "_pr_plan_after.txt", update.queryExecution.executedPlan.toString)
          writeText(currentRun + "_pr_logical_plan.txt", update.queryExecution.toString)
        }
        messages.unpersist(blocking = true)
        state.unpersist(blocking = true)
        next
      }
    }
    validateResult("pagerank", state.select("id", "rank").as[(Long, Double)].collect().toMap, expectedRanks)
    state.unpersist(blocking = true); weightedEdges.unpersist(blocking = true)
    g.vertices.unpersist(blocking = true); g.edges.unpersist(blocking = true)
  }

## 6. Kiểm tra đúng trước khi đo tốc độ

Reference là vòng lặp Scala trên driver, độc lập với GraphX và GraphFrames.
So incoming amount tại từng vertex, không chỉ tổng tiền toàn graph.
So PageRank tại từng vertex sau đúng số vòng, đồng thời kiểm tra rank mass bằng 1.
Reference collect toàn bộ graph nên dùng cho quy mô vừa; chi phí này nằm ngoài timer benchmark.

In [16]:
def reference(): Unit = {
    // Independent driver reference, excluded from all benchmark windows.
    val ids = vertices.select("id").as[Long].collect().sorted
    val pos = ids.zipWithIndex.toMap
    val e = edges.select("src", "dst", "amount").as[(Long, Long, Double)].collect()
    val incoming = Array.fill(ids.length)(0.0)
    val degree = Array.fill(ids.length)(0L)
    e.foreach { case (src, dst, amount) => incoming(pos(dst)) += amount; degree(pos(src)) += 1 }
    expectedIncoming = ids.indices.map(i => ids(i) -> incoming(i)).toMap
    var ranks = Array.fill(ids.length)(1.0 / n)
    (1 to iterations).foreach { _ =>
      val dangling = ids.indices.filter(i => degree(i) == 0).map(ranks(_)).sum
      val next = Array.fill(ids.length)((1.0 - damping) / n + damping * dangling / n)
      e.foreach { case (src, dst, _) => next(pos(dst)) += damping * ranks(pos(src)) / degree(pos(src)) }
      ranks = next
    }
    expectedRanks = ids.indices.map(i => ids(i) -> ranks(i)).toMap
  }

In [17]:
reference()
println(s"Reference ready: ${expectedRanks.size} vertex ranks")

collect at cmd16.sc:3

collect at cmd16.sc:5

Reference ready: 54019 vertex ranks


### Microbenchmark serialize/deserialize payload

Cùng 256 cạnh, cùng thứ tự và cùng Kryo serializer, đo Array[Edge[Double]] và Array[InternalRow].
Đây là probe trên driver, không phải toàn bộ chi phí shuffle, EdgePartition hay columnar cache.
Python sẽ trình bày phép đo này riêng với task deserialize/result serialize metrics.

In [18]:
def serializationProbe(): Unit = {
    // A controlled driver microbenchmark, not a measurement of entire Spark shuffle cost.
    val rows = edges.orderBy("edge_id").select("src", "dst", "amount").limit(256)
    val graphxPayload = rows.as[(Long, Long, Double)].collect().map { case (src, dst, amount) => Edge(src, dst, amount) }
    val sqlPayload = rows.queryExecution.toRdd.collect().map(_.copy())
    val writer = new PrintWriter(new File(output, "serialization_probe.csv"))
    csv(writer, Seq("representation", "repeat", "records", "loops", "payload_bytes", "serialize_ms", "deserialize_ms", "gc_ms"))
    def probe[A: scala.reflect.ClassTag](name: String, payload: A): Unit = {
      val serializer = org.apache.spark.SparkEnv.get.serializer.newInstance()
      (0 until 30).foreach { _ => serializer.deserialize[A](serializer.serialize(payload)) }
      (0 until 5).foreach { repeat =>
        val gcStart = gcMillis
        val start = System.nanoTime()
        var buffer = serializer.serialize(payload)
        (1 until 200).foreach { _ => buffer = serializer.serialize(payload) }
        val serializedMs = (System.nanoTime() - start) / 1e6
        val startDecode = System.nanoTime()
        var decoded = serializer.deserialize[A](buffer.duplicate())
        (1 until 200).foreach { _ => decoded = serializer.deserialize[A](buffer.duplicate()) }
        val decodeMs = (System.nanoTime() - startDecode) / 1e6
        require(decoded != null)
        csv(writer, Seq(name, repeat, graphxPayload.length, 200, buffer.limit(), serializedMs, decodeMs, gcMillis - gcStart))
      }
    }
    try {
      probe("GraphX Edge[Double] array", graphxPayload)
      probe("Spark SQL InternalRow array", sqlPayload)
    } finally writer.close()
  }

In [19]:
serializationProbe()
println("Serialization probe saved")

collect at cmd18.sc:4

toRdd at cmd18.sc:5

collect at cmd18.sc:5

Serialization probe saved


## 7. Ma trận benchmark

GraphX dùng RandomVertexCut, EdgePartition1D, EdgePartition2D và CanonicalRandomVertexCut.
GraphFrames dùng src/dst nhân với AQE OFF/ON.
Mỗi cấu hình có một warmup, sau đó đo ba lần; thứ tự trộn bằng seed cố định.
Giữ một Spark JVM, cùng cache policy và cùng input; dọn cache của benchmark sau mỗi run.
Broadcast join tắt để quan sát join/shuffle, nên kết quả không đại diện cho mọi cấu hình SQL mặc định.

In [20]:
val configs = Seq(
  ("GraphX", "RandomVertexCut", false),
  ("GraphX", "EdgePartition1D", false),
  ("GraphX", "EdgePartition2D", false),
  ("GraphX", "CanonicalRandomVertexCut", false),
  ("GraphFrames", "src", false), ("GraphFrames", "dst", false),
  ("GraphFrames", "src", true), ("GraphFrames", "dst", true)
)
val random = new scala.util.Random(seed)
val schedule = new PrintWriter(new File(output, "schedule.csv"))
csv(schedule, Seq("run", "framework", "strategy", "aqe", "repeat", "warmup", "order"))
var order = 0
var completedRuns = 0

completedRuns: Int = 0

### Ghi Spark UI và metadata

UI có thể mở trực tiếp bằng địa chỉ in ở phần khởi tạo Spark.
HTML snapshot được ghi ở warmup cho cả GraphX và GraphFrames; đây không phải screenshot browser.
Event log giữ toàn bộ các run để replay Jobs, Stages, Storage và SQL bằng Spark History Server.
Metadata ghi phiên bản, heap JVM, cấu hình và input được chọn để Python không trộn hai thí nghiệm.

In [21]:
def captureUi(framework: String): Unit = {
  val directory = new File(output, "ui"); directory.mkdirs()
  sc.uiWebUrl.foreach { base =>
    Seq("jobs/", "stages/", "storage/", "SQL/").foreach { endpoint =>
      try {
        val source = scala.io.Source.fromURL(base + "/" + endpoint)
        try {
          val writer = new PrintWriter(new File(directory, framework + "_" + endpoint.stripSuffix("/") + ".html"))
          try writer.write(source.mkString) finally writer.close()
        } finally source.close()
      } catch { case error: Exception => println(s"UI snapshot: ${error.getMessage}") }
    }
  }
}
val mapper = new com.fasterxml.jackson.databind.ObjectMapper()
def writeJson(file: File, fields: (String, Any)*): Unit =
  mapper.writerWithDefaultPrettyPrinter().writeValue(file, fields.toMap.asJava)

writeJson(new File(output, "input_manifest.json"),
  "source" -> rawPath.getAbsolutePath, "currency" -> currency,
  "amount_column" -> "Amount Received", "eligible_rows" -> eligibleCount,
  "sample_edges" -> sampleEdges, "seed" -> seed, "vertices" -> n, "edges" -> edgeCount,
  "vertices_path" -> new File(inputDir, "vertices").getAbsolutePath,
  "edges_path" -> new File(inputDir, "edges").getAbsolutePath,
  "id_encoding" -> "1-based index of sorted (bank, account) pairs",
  "sampling" -> "Spark rand(seed) ordering over all eligible USD edges")
writeText("environment.txt", s"""spark=${spark.version}
scala=${scala.util.Properties.versionNumberString}
java=${System.getProperty("java.version")}
master=${sc.master}
vertices=$n
edges=$edgeCount
partitions=$p
iterations=$iterations
repeats=$repeats
warmups=$warmups
heap_max_bytes=${Runtime.getRuntime.maxMemory()}
execution=native Scala notebook
""")

In [22]:
val captured = scala.collection.mutable.Set.empty[String]
try {
  (-warmups until repeats).foreach { repeat =>
    random.shuffle(configs).foreach { case (framework, strategy, aqe) =>
      currentRun = s"${framework}_${strategy}_aqe${aqe}_r$repeat"
      csv(schedule, Seq(currentRun, framework, strategy, aqe, repeat, repeat < 0, order))
      order += 1
      cleanup()
      spark.conf.set("spark.sql.adaptive.enabled", "false")
      if (framework == "GraphX") runGraphX(strategy, repeat == 0)
      else runGraphFrames(strategy, aqe, repeat == 0)
      cleanup()
      if (repeat < 0 && !captured(framework)) {
        captureUi(framework)
        captured += framework
      }
      completedRuns += 1
      println(s"Passed $order/${configs.size * (warmups + repeats)}: $currentRun")
    }
  }
} finally {
  metrics.close(); checks.close(); loads.close(); schedule.close(); progress.close()
}

map at cmd10.sc:9

map at GraphImpl.scala:100

mapPartitions at VertexRDD.scala:356

mapPartitions at VertexRDD.scala:356

fold at VertexRDDImpl.scala:90

...

mapPartitions at GraphImpl.scala:207

sum at cmd14.sc:39

sum at cmd14.sc:39

collect at cmd14.sc:48

Passed 32/32: GraphX_EdgePartition1D_aqefalse_r2


## 8. Đóng event log, công bố kết quả và chuyển sang Python

Chỉ cập nhật latest_run.json sau khi mọi cấu hình vượt qua correctness check.
Dừng Spark để event log hoàn tất, tránh Python đọc một lần chạy đang dở.
Notebook tiếp theo chỉ đọc output, không chạy lại Spark hoặc thay đổi workload.

In [23]:
require(completedRuns == configs.size * (warmups + repeats), "Benchmark chưa hoàn tất.")
writeText("SUCCESS", "All configurations matched independent incoming-sum and PageRank references.")
vertices.unpersist(blocking = true)
edges.unpersist(blocking = true)
spark.stop()
writeJson(new File(reportRoot, "latest_run.json"),
  "run_dir" -> root.toPath.relativize(output.toPath).toString,
  "workflow" -> "native-scala-notebook",
  "benchmark_notebook" -> "01_scala_graph_benchmark.ipynb")
println(s"Completed: ${configs.size * (warmups + repeats)} runs, ${2 * configs.size * (warmups + repeats)} correctness checks")
println(s"Output: ${output.getAbsolutePath}")
println("Next: open 02_python_visualize.ipynb and Run All.")

Completed: 32 runs, 64 correctness checks


Output: /home/tuanta/graphX/reports/architecture/runs/scala_1792035232036


Next: open 02_python_visualize.ipynb and Run All.


Các bảng runtime thô không đủ để kết luận framework nào tốt hơn.
Notebook Python sẽ đối chiếu runtime với cache, GC, shuffle, CPU, stage/task, skew và plan.
Trong local mode không có network giữa các máy; durable checkpoint có chi phí I/O và có thể thay đổi index identity.
Các giới hạn đó được giữ rõ trong phần kết luận, thay vì suy rộng từ một phép đo.